# Run a trained policy

This notebook will provide examples on how to run a trained policy and visualize the rollout.

In [10]:
import argparse
import json
import h5py
import imageio
import numpy as np
import os
from copy import deepcopy

import torch

import robomimic
import robomimic.utils.file_utils as FileUtils
import robomimic.utils.torch_utils as TorchUtils
import robomimic.utils.tensor_utils as TensorUtils
import robomimic.utils.obs_utils as ObsUtils
from robomimic.envs.env_base import EnvBase
from robomimic.algo import RolloutPolicy

import urllib.request


### Download policy checkpoint
First, let's try downloading a pretrained model from our model zoo.

In [11]:
# Get pretrained checkpooint from the model zoo

ckpt_path = "lift_ph_low_dim_epoch_1000_succ_100.pth"
# Lift (Proficient Human)
urllib.request.urlretrieve(
    "http://downloads.cs.stanford.edu/downloads/rt_benchmark/model_zoo/lift/bc_rnn/lift_ph_low_dim_epoch_1000_succ_100.pth",
    filename=ckpt_path
)

assert os.path.exists(ckpt_path)

### Loading trained policy
We have a convenient function called `policy_from_checkpoint` that takes care of building the correct model from the checkpoint and load the trained weights. Of course you could also load the checkpoint manually.

In [12]:
device = TorchUtils.get_torch_device(try_to_use_cuda=True)

# restore policy
policy, ckpt_dict = FileUtils.policy_from_checkpoint(ckpt_path=ckpt_path, device=device, verbose=True)

============= Loaded Config =============
{
    "algo_name": "bc",
    "experiment": {
        "name": "core_bc_rnn_lift_ph_low_dim",
        "validate": true,
        "logging": {
            "terminal_output_to_txt": true,
            "log_tb": true
        },
        "save": {
            "enabled": true,
            "every_n_seconds": null,
            "every_n_epochs": 50,
            "epochs": [],
            "on_best_validation": false,
            "on_best_rollout_return": false,
            "on_best_rollout_success_rate": true
        },
        "epoch_every_n_steps": 100,
        "validation_epoch_every_n_steps": 10,
        "env": null,
        "additional_envs": null,
        "render": false,
        "render_video": true,
        "keep_all_videos": false,
        "video_skip": 5,
        "rollout": {
            "enabled": true,
            "n": 50,
            "horizon": 400,
            "rate": 50,
            "warmstart": 0,
            "terminate_on_success": true
     

### Creating rollout envionment
The policy checkpoint also contains sufficient information to recreate the environment that it's trained with. Again, you may manually create the environment.

In [13]:
# create environment from saved checkpoint
env, _ = FileUtils.env_from_checkpoint(
    ckpt_dict=ckpt_dict, 
    render=False, # we won't do on-screen rendering in the notebook
    render_offscreen=False, # render to RGB images for video
    verbose=True,
)

Created environment with name Lift
Action size is 7
ROBOMIMIC WARNING(
    No environment version found in dataset!
    Cannot verify if dataset and installed environment versions match
)
============= Loaded Environment =============
Lift
{
    "camera_depths": false,
    "camera_heights": 84,
    "camera_widths": 84,
    "control_freq": 20,
    "controller_configs": {
        "control_delta": true,
        "damping": 1,
        "damping_limits": [
            0,
            10
        ],
        "impedance_mode": "fixed",
        "input_max": 1,
        "input_min": -1,
        "interpolation": null,
        "kp": 150,
        "kp_limits": [
            0,
            300
        ],
        "orientation_limits": null,
        "output_max": [
            0.05,
            0.05,
            0.05,
            0.5,
            0.5,
            0.5
        ],
        "output_min": [
            -0.05,
            -0.05,
            -0.05,
            -0.5,
            -0.5,
            -0

### Define the rollout loop
Now let's define the main rollout loop. The loop runs the policy to a target `horizon` and optionally writes the rollout to a video.

In [ ]:
def rollout(policy, env, horizon, render=False, video_writer=None, video_skip=5, camera_names=None, replica_exchange=False, n_replicas=1, lam_start=None):
    """
    Helper function to carry out rollouts. Supports on-screen rendering, off-screen rendering to a video, 
    and returns the rollout trajectory.
    Args:
        policy (instance of RolloutPolicy): policy loaded from a checkpoint
        env (instance of EnvBase): env loaded from a checkpoint or demonstration metadata
        horizon (int): maximum horizon for the rollout
        render (bool): whether to render rollout on-screen
        video_writer (imageio writer): if provided, use to write rollout to video
        video_skip (int): how often to write video frames
        camera_names (list): determines which camera(s) are used for rendering. Pass more than
            one to output a video with multiple camera views concatenated horizontally.
        replica_exchange (bool): whether to enable replica exchange for diffusion policy inference
        n_replicas (int): number of replicas to use for replica exchange
        lam_start (float): lambda start value for replica exchange
    Returns:
        stats (dict): some statistics for the rollout - such as return, horizon, and task success
    """
    assert isinstance(env, EnvBase)
    assert isinstance(policy, RolloutPolicy)
    assert not (render and (video_writer is not None))

    if replica_exchange:
        if not hasattr(policy, "set_replica_exchange"):
            print(
                "Warning: replica_exchange=True passed to rollout, but the policy object may not support "
                "direct replica exchange configuration."
            )

    policy.start_episode()
    obs = env.reset()
    state_dict = env.get_state()

    # hack that is necessary for robosuite tasks for deterministic action playback
    obs = env.reset_to(state_dict)

    results = {}
    video_count = 0  # video frame counter
    total_reward = 0.
    try:
        for step_i in range(horizon):

            # get action from policy
            act = policy(ob=obs)

            # play action
            next_obs, r, done, _ = env.step(act)

            # compute reward
            total_reward += r
            success = env.is_success()["task"]

            # visualization
            if render:
                env.render(mode="human", camera_name=camera_names[0])
            if video_writer is not None:
                if video_count % video_skip == 0:
                    video_img = []
                    for cam_name in camera_names:
                        video_img.append(env.render(mode="rgb_array", height=512, width=512, camera_name=cam_name))
                    video_img = np.concatenate(video_img, axis=1) # concatenate horizontally
                    video_writer.append_data(video_img)
                video_count += 1

            # break if done or if success
            if done or success:
                break

            # update for next iter
            obs = deepcopy(next_obs)
            state_dict = env.get_state()

    except env.rollout_exceptions as e:
        print("WARNING: got rollout exception {}".format(e))

    stats = dict(Return=total_reward, Horizon=(step_i + 1), Success_Rate=float(success))

    return stats


### Run the policy
Now let's rollout the policy!

In [15]:
rollout_horizon = 400
np.random.seed(0)
torch.manual_seed(0)
video_path = "rollout.mp4"
video_writer = imageio.get_writer(video_path, fps=20)

# replica exchange settings
replica_exchange = True
n_replicas = 2
lam_start = 0.98

In [16]:
stats = rollout(
    policy=policy, 
    env=env, 
    horizon=rollout_horizon, 
    render=False, 
    video_writer=None, 
    video_skip=5, 
    camera_names=["agentview"],
    replica_exchange=replica_exchange,
    n_replicas=n_replicas,
    lam_start=lam_start,
)
print(stats)
video_writer.close()

ObservationKeyToModalityDict: robot0_joint_pos not found, adding robot0_joint_pos to mapping with assumed low_dim modality!
ObservationKeyToModalityDict: robot0_joint_pos_cos not found, adding robot0_joint_pos_cos to mapping with assumed low_dim modality!
ObservationKeyToModalityDict: robot0_joint_pos_sin not found, adding robot0_joint_pos_sin to mapping with assumed low_dim modality!
ObservationKeyToModalityDict: robot0_joint_vel not found, adding robot0_joint_vel to mapping with assumed low_dim modality!
ObservationKeyToModalityDict: robot0_eef_vel_lin not found, adding robot0_eef_vel_lin to mapping with assumed low_dim modality!
ObservationKeyToModalityDict: robot0_eef_vel_ang not found, adding robot0_eef_vel_ang to mapping with assumed low_dim modality!
ObservationKeyToModalityDict: robot0_gripper_qvel not found, adding robot0_gripper_qvel to mapping with assumed low_dim modality!
{'Return': 1.0, 'Horizon': 59, 'Success_Rate': 1.0}


In [17]:
# Run multiple rollouts and evaluate the results
n_trials = 5
results = []
for trial in range(n_trials):
    np.random.seed(trial)
    torch.manual_seed(trial)

    stats = rollout(
        policy=policy,
        env=env,
        horizon=rollout_horizon,
        render=False,
        video_writer=None,
        video_skip=5,
        camera_names=["agentview"],
    )
    results.append(stats)
    print(f"Trial {trial + 1}/{n_trials}: {stats}")

# Aggregate evaluation metrics
returns = [r["Return"] for r in results]
horizons = [r["Horizon"] for r in results]
success_rates = [r["Success_Rate"] for r in results]

print("\nSummary:")
print(f"  mean return: {np.mean(returns):.3f}  std: {np.std(returns):.3f}")
print(f"  mean horizon: {np.mean(horizons):.1f}  std: {np.std(horizons):.1f}")
print(f"  mean success rate: {np.mean(success_rates):.3f}  std: {np.std(success_rates):.3f}")

# Save results for later analysis
results_path = "rollout_results.json"
with open(results_path, "w") as f:
    json.dump(results, f, indent=2)
print(f"Saved rollout results to {results_path}")

Trial 1/5: {'Return': 1.0, 'Horizon': 59, 'Success_Rate': 1.0}
Trial 2/5: {'Return': 1.0, 'Horizon': 38, 'Success_Rate': 1.0}
Trial 3/5: {'Return': 1.0, 'Horizon': 38, 'Success_Rate': 1.0}
Trial 4/5: {'Return': 1.0, 'Horizon': 42, 'Success_Rate': 1.0}
Trial 5/5: {'Return': 1.0, 'Horizon': 41, 'Success_Rate': 1.0}

Summary:
  mean return: 1.000  std: 0.000
  mean horizon: 43.6  std: 7.9
  mean success rate: 1.000  std: 0.000
Saved rollout results to rollout_results.json


### Visualize the rollout

In [18]:
# from IPython.display import Video
# Video(video_path)